# 08 - End-to-End Classical Attire Demonstration

This is the standalone ScreenCam notebook. It takes one Fashionpedia image through loading, preprocessing, garment detection, segmentation, handcrafted feature extraction, target recognition, transparent rule evaluation, and an **appropriate**, **inappropriate**, or **review required** result. It shows every reason and evidence region. No deep learning or GPU model is used.

In [ ]:
import csv,json,sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
cwd=Path.cwd().resolve(); REPO=next((p for p in [cwd,*cwd.parents] if (p/'implementation'/'src').exists()),None)
if REPO is None: raise RuntimeError('Start Jupyter inside the repository.')
sys.path.insert(0,str(REPO/'implementation'/'src'))
from ipcv_attire.dataset_policy import load_policy,require_showcase_approval
from ipcv_attire.features import extract_handcrafted_features
from ipcv_attire.pipeline import AttirePipeline
from ipcv_attire.preprocessing import preprocess_image
from ipcv_attire.training import train_pipeline
from ipcv_attire.visualization import draw_detections,draw_masks,render_report_overlay
DATA=REPO/'implementation'/'data'; OUTPUT=REPO/'implementation'/'outputs'/'predictions'/'notebook-demo'; POLICY=load_policy(DATA/'dataset-policy.json')


## Configuration

The laptop default builds a deterministic smoke bundle only when missing. On the 32 GB desktop, set `TRAINING_PROFILE='full'` once; later demonstrations load the frozen bundle quickly. `FORCE_RETRAIN=False` prevents accidental overwriting. Only an approved, non-risk Fashionpedia image can be displayed.

In [ ]:
TRAIN_IF_MISSING=True
FORCE_RETRAIN=False
TRAINING_PROFILE='smoke'  # use 'full' on the desktop
DEMO_IMAGE_ID='13665'
BUNDLE=REPO/'implementation'/'models'/f'classical-attire-{TRAINING_PROFILE}'
if FORCE_RETRAIN: raise RuntimeError('Delete the exact ignored bundle manually before an intentional retrain.')
if not (BUNDLE/'manifest.json').exists():
    if not TRAIN_IF_MISSING: raise FileNotFoundError(f'Missing model bundle: {BUNDLE}')
    train_pipeline(manifest_dir=DATA/'interim'/'manifests',fashionpedia_root=DATA/'raw'/'fashionpedia',policy=POLICY,bundle_dir=BUNDLE,profile_name=TRAINING_PROFILE)
pipeline=AttirePipeline.load(BUNDLE); print('Loaded',BUNDLE); print(json.dumps(pipeline.bundle_metadata,indent=2))


## 1. Safe input selection

This gate checks the generated metadata risk flag and `showcase-manifest.csv`. Changing the ID to an unapproved or risk-flagged Fashionpedia image raises `PermissionError` before pixels are rendered.

In [ ]:
manifest=DATA/'interim'/'manifests'/'fashionpedia-images.csv'
with manifest.open(encoding='utf-8',newline='') as handle: record=next(r for r in csv.DictReader(handle) if r['image_id']==DEMO_IMAGE_ID)
require_showcase_approval(DEMO_IMAGE_ID,DATA/'showcase-manifest.csv',display_risk=record['display_risk']=='1')
image_path=DATA/'raw'/'fashionpedia'/record['relative_image_path']
print({'image_id':DEMO_IMAGE_ID,'split':record['final_split'],'ground_truth_label':record['compliance_label'],'path':str(image_path)})


## 2. Loading and preprocessing

Pillow performs decoding, EXIF orientation correction, grayscale/RGBA conversion, and RGB output. Large inputs are aspect-ratio resized. CLAHE is applied only to LAB luminance to improve local contrast without independently distorting colour channels.

In [ ]:
prepared=preprocess_image(image_path,maximum_side=POLICY['pipeline']['maximum_image_side'],minimum_side=POLICY['pipeline']['minimum_image_side'])
fig,axes=plt.subplots(1,3,figsize=(16,5))
for axis,image,title in zip(axes,[prepared.original_rgb,prepared.resized_rgb,prepared.normalized_rgb],['Original RGB','Bounded resize','CLAHE normalized']): axis.imshow(image); axis.set_title(title); axis.axis('off')
plt.tight_layout(); print('scale=',prepared.scale,'warnings=',prepared.warnings)


## 3. HOG-linear-SVM component detection

Upper-body, bottom-body, footwear, and headwear windows are scored by separate classical linear SVMs. Non-maximum suppression removes duplicate windows. Detection scores and box IDs remain in the report.

In [ ]:
detections=pipeline.detector.detect(prepared.normalized_rgb)
display(pd.DataFrame([d.to_dict() for d in detections]))
plt.figure(figsize=(9,8)); plt.imshow(draw_detections(prepared.normalized_rgb,detections)); plt.axis('off'); plt.title('Component boxes after NMS'); plt.show()


## 4. GrabCut segmentation and outfit grouping

Each box initializes GrabCut, followed by morphological cleanup and largest-connected-component filtering. Components are grouped into outfits by horizontal overlap, centre distance, and vertical order. Bad masks or ambiguous grouping force review.

In [ ]:
report,stages=pipeline.analyze_with_stages(image_path); components=[c for o in report.outfits for c in o.components]
segmentation_rows=[{'outfit':o.outfit_id,'component_id':c.component_id,'type':c.detection.component,'mask_valid':c.segmentation_valid,'mask_area_ratio':c.mask_area_ratio,'edge_contact':c.mask_edge_contact_ratio,'warnings':'; '.join(c.warnings)} for o in report.outfits for c in o.components]
display(pd.DataFrame(segmentation_rows)); plt.figure(figsize=(9,8)); plt.imshow(draw_masks(stages.normalized_rgb,components)); plt.axis('off'); plt.title('GrabCut masks by component'); plt.show()


## 5. Handcrafted feature evidence

For the first valid segmented component, the notebook displays HOG edge structure, uniform-LBP texture codes, and the HSV histogram. The classifier vector additionally contains colour moments, edge density, and mask-shape statistics.

In [ ]:
valid=next((c for c in components if c.segmentation_valid and c.mask is not None),None)
if valid is None: print('No valid segmented component; the final result must require review.')
else:
    x1,y1,x2,y2=valid.detection.bbox; crop=stages.normalized_rgb[y1:y2,x1:x2]; crop_mask=valid.mask[y1:y2,x1:x2]; evidence=extract_handcrafted_features(crop,crop_mask,pipeline.feature_config)
    print(valid.component_id,valid.detection.component,evidence.feature_lengths,'total=',evidence.vector.size)
    fig,axes=plt.subplots(1,4,figsize=(16,3)); axes[0].imshow(crop); axes[0].set_title('Segment crop'); axes[1].imshow(evidence.hog_image,cmap='gray'); axes[1].set_title('HOG'); axes[2].hist(evidence.lbp_image.ravel(),bins=10); axes[2].set_title('Uniform LBP'); axes[3].plot(evidence.hsv_histogram); axes[3].set_title('HSV histogram'); plt.tight_layout()


## 6. Target predictions and uncertainty

Probabilities above a target's high threshold are confident presence, below its low threshold confident absence, and between them uncertain. Footwear/headwear presence uses the corresponding trained component detector as a documented proxy.

In [ ]:
prediction_rows=[]
for outfit in report.outfits:
    for component in outfit.components:
        for p in component.predictions:
            state='present' if p.probability>=p.high_threshold else ('absent' if p.probability<=p.low_threshold else 'uncertain')
            prediction_rows.append({'outfit':outfit.outfit_id,'component':component.component_id,'target':p.target_id,'probability':p.probability,'low':p.low_threshold,'high':p.high_threshold,'state':state})
display(pd.DataFrame(prediction_rows))


## 7. Complete rubric audit

All supported rules are listed even after an immediate failure. Unsupported conditions are explicit audit rows, never inferred. This table is the trace from model evidence to the decision.

In [ ]:
rule_rows=[{'outfit':o.outfit_id,'outfit_decision':o.decision.value,'rule':r.rule_id,'status':r.status.value,'confidence':r.confidence,'severity':r.severity,'supported':r.supported,'evidence_region':r.evidence_region,'reason':r.reason} for o in report.outfits for r in o.rules]
with pd.option_context('display.max_colwidth',120): display(pd.DataFrame(rule_rows))


## 8. Final decision, all reasons, and visual evidence

Per outfit, any confident failure wins; otherwise incomplete required evidence produces review-required. At image level, any failed outfit wins, then any review-required outfit, otherwise all outfits are appropriate.

In [ ]:
plt.figure(figsize=(10,9)); plt.imshow(render_report_overlay(stages.normalized_rgb,report)); plt.axis('off'); plt.title(report.to_dict()['user_facing_decision'].upper()); plt.show()
print('OVERALL:',report.to_dict()['user_facing_decision'].upper(),'-',report.reason)
for outfit in report.outfits:
    print('\n',outfit.outfit_id,outfit.decision.value); print('Passed:',*outfit.passed_reasons,sep='\n- '); print('Failed:',*outfit.failed_reasons,sep='\n- '); print('Review:',*outfit.review_reasons,sep='\n- '); print('Unsupported:',*outfit.unsupported_reasons,sep='\n- ')
print('\nTimings (ms):',json.dumps(report.timings_ms,indent=2)); print('Warnings:',report.warnings)


## 9. Export the same auditable result

The CLI and public Python API return the same JSON structure. Export is permitted here because the selected Fashionpedia image passed the presentation gate.

In [ ]:
OUTPUT.mkdir(parents=True,exist_ok=True); report_path=OUTPUT/f'{DEMO_IMAGE_ID}-report.json'; overlay_path=OUTPUT/f'{DEMO_IMAGE_ID}-overlay.png'
report_path.write_text(json.dumps(report.to_dict(),indent=2)+'\n',encoding='utf-8'); Image.fromarray(render_report_overlay(stages.normalized_rgb,report)).save(overlay_path)
print('Saved',report_path); print('Saved',overlay_path)


## Public interface

```python
pipeline = AttirePipeline.load(model_bundle)
report = pipeline.analyze(image_path_or_array)
```

For arbitrary external input the API runs without a showcase gate; the gate applies when coursework renders Fashionpedia images. Conclusions remain limited to the Fashionpedia domain and do not claim real-university validation.